# Guardrails AI: validation patterns

A hands-on walkthrough of validating LLM text, handling failures, extracting structured data, and integrating Guardrails with LangChain.

Run the notebook from top to bottom. Later examples reuse the model configuration, constants, and validators defined in earlier cells.

## Contents

1. [Setup and configuration](#1-setup-and-configuration)
2. [PII detection](#2-pii-detection)
3. [Competitor checks](#3-competitor-check)
4. [Toxic language](#4-toxic-language)
5. [Topic restriction](#5-restrict-to-topic)
6. [Failure actions](#6-onfailactions)
7. [Structured extraction](#7-io-structure-validation-using-guardrailsai)
8. [LangChain integration](#8-langchain-x-guardrails-ai)
9. [Streaming validation](#9-guardrails-validation-during-streaming)

In [67]:
print("Hello, This is Guardian AI, your personal assistant. How can I help you today?")

Hello, This is Guardian AI, your personal assistant. How can I help you today?


## 1. Setup and configuration

Use the project virtual environment, set the Groq API key when prompted, and configure the model and assistant instructions before running the examples.

In [68]:
import os, getpass
from pathlib import Path

In [69]:
_root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)
print("Working dir:", os.getcwd())

Working dir: h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI


In [70]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

In [71]:
BOT_MODEL = "groq/openai/gpt-oss-120b"  # litellm needs the groq/ provider prefix
SYSTEM_PROMPT = (
    "You are NimbusPay's customer-support assistant. "
    "NimbusPay is a digital payments app: cards, wallets, refunds, account help. "
    "Be concise, friendly, and accurate."
)

print("Config ready:", BOT_MODEL)

Config ready: groq/openai/gpt-oss-120b


## 2. PII Detection 

In [72]:
from guardrails_ai.detect_pii import DetectPII
from guardrails import Guard, OnFailAction

In [73]:
pii_guard = Guard().use(
    DetectPII(
        pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
        on_fail=OnFailAction.FIX,
        use_local=True,     # light: Presidio + spaCy small, no torch
    )
)

In [74]:
raw = "We'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com."
out = pii_guard.validate(raw)
print(out)

ValidationOutcome[TypeVar](
    call_id='2994747370640',
    raw_llm_output="We'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.",
    validation_summaries=[
        ValidationSummary(
            validator_name='DetectPII',
            validator_status='fail',
            property_path='$',
            failure_reason="The following text in your response contains PII:\nWe'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.",
            error_spans=[
                ErrorSpan(
                    start=18,
                    end=37,
                    reason='PII detected in 4111 1111 1111 1111'
                ),
                ErrorSpan(
                    start=63,
                    end=75,
                    reason='PII detected in jane@example'
                )
            ]
        )
    ],
    validated_output="We'll refund card <CREDIT_CARD> and email the receipt to <EMAIL_ADDRESS>.",
    reask=None,
    validation_

h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [75]:
print("IN :", raw)
print("OUT:", out.validated_output)

IN : We'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.
OUT: We'll refund card <CREDIT_CARD> and email the receipt to <EMAIL_ADDRESS>.


## 3. Competitor Check

The next examples compare a response with no competitor mentions to one that names configured competitors.

In [76]:
from guardrails_ai.competitor_check import CompetitorCheck
COMPETITORS = ["Stripe", "PayPal", "Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
    "and PayPal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."
)

### 3.1 Passing example: no competitor mentions

In [77]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FILTER, use_local=True)
)
out = comp_guard.validate(GOOD_TEXT)
print("OUT:", repr(out.validated_output))

OUT: "Thanks for reaching out! We're happy to help with your refund."


h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


### 3.2 Failing example: competitor mentions

In [78]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FIX, use_local=True)
)
out = comp_guard.validate(BAD_TEXT)
print("OUT:", repr(out.validated_output))

OUT: "Thanks for reaching out!Honestly, [COMPETITOR] has lower fees than us, and [COMPETITOR] is easier to set up.But we're happy to help with your refund."


h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


## 4. Toxic Language

In [79]:
from guardrails_ai.toxic_language import ToxicLanguage

In [80]:
tox_guard = Guard().use(
    ToxicLanguage(
        threshold=0.5,
        validation_method="sentence",
        on_fail=OnFailAction.FIX,
        use_local=True, 
    )
)

In [81]:
rude = "You are completely useless and your stupid fuck app stole my money."
out = tox_guard.validate(rude)
print("validation_passed:", out.validation_passed)
print("OUT:", repr(out.validated_output))

validation_passed: True
OUT: ''


h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [82]:
out

ValidationOutcome[TypeVar](call_id='2997013478320', raw_llm_output='You are completely useless and your stupid fuck app stole my money.', validation_summaries=[ValidationSummary(validator_name='ToxicLanguage', validator_status='fail', property_path='$', failure_reason='The following sentences in your response were found to be toxic:\n\n- You are completely useless and your stupid fuck app stole my money.', error_spans=[ErrorSpan(start=0, end=67, reason='Toxic language detected: toxicity, obscene, insult')])], validated_output='', reask=None, validation_passed=True, error=None)

## 5. Restrict to Topic

In [83]:
import json, litellm
from guardrails_ai.restricttotopic import RestrictToTopic
from guardrails.errors import ValidationError

In [84]:
def groq_topic_classifier(text, topics):
    resp = litellm.completion(
        model=BOT_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": (
            'Return ONLY a JSON object {"topics_present": [...]} listing which of these '
            f'topics appear in the text.\nTopics: {topics}\nText: "{text}"'
        )}],
    )

    payload = json.loads(resp.choices[0].message.content)
    if not isinstance(payload, dict):
        raise ValueError("Topic classifier response must be a JSON object.")

    topics_present = payload.get("topics_present")
    if not isinstance(topics_present, list) or not all(
        isinstance(topic, str) for topic in topics_present
    ):
        raise ValueError("Topic classifier response must include a list of topic names.")

    return topics_present

In [85]:
topic_guard = Guard().use(
    RestrictToTopic(
        valid_topics=["payments", "refunds", "accounts", "cards", "wallets"],
        invalid_topics=["politics", "medical advice", "investing tips"],
        disable_classifier=True,             # skip the transformer; no local/remote model needed
        disable_llm=False,
        llm_callable=groq_topic_classifier,  # Groq instead of OpenAI gpt-4o
        on_fail=OnFailAction.EXCEPTION,
    )
)



ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.
Device set to use cpu


In [86]:
off_topic = "Forget payments - who do you think will win the next presidential election?"

In [87]:
out1 = None
try:
    out1 = topic_guard.validate(off_topic)
    print(out1)
    print("Passed (unexpected):", out1.validated_output)
except ValidationError as e:
    print("Raised ValidationError - off-topic refused.")
    print(str(e)[:300])


h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


Raised ValidationError - off-topic refused.
Validation failed for field with errors: Invalid topics found: ['politics']


## 6. OnFailActions

We are performing Multiple OnFailActions on CompetitorCheck validator

from guardrails import Guard, OnFailAction
from guardrails_ai.competitor_check import CompetitorCheck
from guardrails.errors import ValidationError

In [88]:
from guardrails import Guard, OnFailAction
from guardrails_ai.competitor_check import CompetitorCheck
from guardrails.errors import ValidationError

In [89]:
from guardrails_ai.competitor_check import CompetitorCheck
COMPETITORS = ["Stripe", "PayPal", "Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
    "and PayPal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."
)

In [90]:
def competitor_guard(action):
    return Guard().use(
        CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    )

print("Input we'll validate five ways:\n", BAD_TEXT)

Input we'll validate five ways:
 Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.


### 6.1 FIX

In [91]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(GOOD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.


Output Raw response ValidationOutcome[TypeVar](
    call_id='2995888870688',
    raw_llm_output="Thanks for reaching out! We're happy to help with your refund.",
    validation_summaries=[],
    validated_output="Thanks for reaching out! We're happy to help with your refund.",
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed: True
OUTPUT:
 Thanks for reaching out! We're happy to help with your refund.


h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [92]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='2997318571088',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output="Thanks for reaching out!Honestly, [COMPETITOR] has lower fees th

h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


### 6.2 FILTER

In [93]:
guard = competitor_guard(OnFailAction.FILTER)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.


Output Raw response ValidationOutcome[TypeVar](
    call_id='2997349995216',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None

h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


### 6.3 REFRAIN

In [94]:
guard = competitor_guard(OnFailAction.REFRAIN)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='2997411754544',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None

h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


### 6.4 REFRAIN vs FILTER

In [95]:
from typing import Optional
from pydantic import BaseModel, Field

STRUCTURED_JSON = (
    '{"answer": "Refunds settle within 24 hours.",'
    ' "comparison": "Honestly Stripe is cheaper than us.",'   # <- this field names a competitor
    ' "category": "refund"}'
)

In [96]:
def structured_guard(action):
    cc = CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    class SupportReply(BaseModel):
        answer: str
        comparison: Optional[str] = Field(default=None, json_schema_extra={"validators": [cc]})
        category: str
    return Guard.for_pydantic(SupportReply)

In [97]:
for action in (OnFailAction.FILTER, OnFailAction.REFRAIN):
    guard = structured_guard(action)
    guard.parse(STRUCTURED_JSON)
    it = guard.history[-1].iterations[-1]
    print(f"=== {action} ===")
    print("  after parsing (sentinel placed in the bad field):")
    print("   ", it.parsed_output)
    print("  guarded_output (what the guard actually returns):")
    print("   ", it.guarded_output)
    print()

h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


=== OnFailAction.FILTER ===
  after parsing (sentinel placed in the bad field):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': <guardrails.actions.filter.Filter object at 0x000002B96CA6A780>, 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {'answer': 'Refunds settle within 24 hours.', 'category': 'refund'}



h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


=== OnFailAction.REFRAIN ===
  after parsing (sentinel placed in the bad field):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': <guardrails.actions.refrain.Refrain object at 0x000002B982436EA0>, 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {}



### 6.5 EXCEPTION

In [98]:
guard = competitor_guard(OnFailAction.EXCEPTION)
try:
    out = guard.validate(BAD_TEXT)
    print("Raw Output", out)
    print("Passed (unexpected):", out.validated_output)
except ValidationError as e:
    print("Raised ValidationError - competitor mentions were rejected.")
    print(str(e)[:300])

Raised ValidationError - competitor mentions were rejected.
Validation failed for field with errors: Found the following competitors: Stripe, PayPal. Please avoid naming those competitors next time


### 6.6 REASK

In [99]:



reask_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.REASK, use_local=True)
)
res = reask_guard(
    model = BOT_MODEL,
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "A customer asks why they should pick NimbusPay. In your answer, explicitly "
            "compare us to Stripe and PayPal by name."
        )},
    ],
    temperature = 0.3,
    num_reasks=2,
)

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.
h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(
h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [100]:
print("FINAL OUTPUT:\n", res.validated_output)

FINAL OUTPUT:
 Sure! Here’s an updated overview of why many businesses choose **NimbusPay**—without referencing specific competitors:

| Feature | NimbusPay |
|---------|-----------|
| **All‑in‑One Dashboard** | Unified view of cards, wallets, refunds, and account settings in a single, easy‑to‑navigate console. |
| **Lower Transaction Fees** | 1.9 % + $0.15 per transaction (no hidden surcharges). |
| **Instant Payouts** | Funds can be transferred to your bank or linked card within minutes, 24/7. |
| **Customizable Checkout** | Drag‑and‑drop builder plus full API control; white‑label branding is included at no extra cost. |
| **Multi‑Currency Support** | Accept and settle in over 150 currencies with competitive FX rates. |
| **Dedicated Support** | 24/7 live chat + phone support for all plans; priority SLA for Enterprise. |
| **Security & Compliance** | PCI‑DSS Level 1, tokenization, built‑in fraud detection, and GDPR‑ready. |
| **Developer Friendly** | Well‑documented SDKs for iOS, And

In [101]:
print(res)

ValidationOutcome[TypeVar](
    call_id='2996909707056',
    raw_llm_output='Sure! Here’s an updated overview of why many businesses choose **NimbusPay**—without referencing specific competitors:\n\n| Feature | NimbusPay |\n|---------|-----------|\n| **All‑in‑One Dashboard** | Unified view of cards, wallets, refunds, and account settings in a single, easy‑to‑navigate console. |\n| **Lower Transaction Fees** | 1.9\u202f%\u202f+\u202f$0.15 per transaction (no hidden surcharges). |\n| **Instant Payouts** | Funds can be transferred to your bank or linked card within minutes, 24/7. |\n| **Customizable Checkout** | Drag‑and‑drop builder plus full API control; white‑label branding is included at no extra cost. |\n| **Multi‑Currency Support** | Accept and settle in over 150 currencies with competitive FX rates. |\n| **Dedicated Support** | 24/7 live chat + phone support for all plans; priority SLA for Enterprise. |\n| **Security & Compliance** | PCI‑DSS Level\u202f1, tokenization, built‑in fra

In [102]:
iters = reask_guard.history[-1].iterations
print("\nLLM round-trips:", len(iters), "| Reask occurred:", len(iters) > 1)

# Show what the model actually returned on EACH attempt (raw, pre-validation):
for i, it in enumerate(iters, start=1):
    print(f"\n--- Attempt {i} (status: {it.status}) ---")
    print(repr(it.raw_output))


LLM round-trips: 2 | Reask occurred: True

--- Attempt 1 (status: fail) ---
'Sure! Here’s why many businesses choose **NimbusPay** over Stripe and PayPal:\n\n| Feature | NimbusPay | Stripe | PayPal |\n|---------|-----------|--------|--------|\n| **All‑in‑One Dashboard** | Unified view of cards, wallets, refunds, and account settings in a single, easy‑to‑navigate console. | Separate UI for payments, billing, and Connect; can feel fragmented. | Separate merchant and consumer dashboards; less cohesive for business ops. |\n| **Lower Transaction Fees** | 1.9\u202f%\u202f+\u202f$0.15 per transaction (no hidden surcharges). | 2.9\u202f%\u202f+\u202f$0.30 per transaction (plus extra for international). | 2.9\u202f%\u202f+\u202f$0.30 per transaction; higher fees for cross‑border and currency conversion. |\n| **Instant Payouts** | Funds can be transferred to your bank or linked card within minutes, 24/7. | Payouts typically take 2–7 business days. | Standard payouts are 1–3 business days; insta

## 7. I/O Structure Validation using GuardrailsAI

In [103]:
import os, getpass
from pathlib import Path

# Hop to the project root (where ./.guardrails lives)
_root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)


### 7.1 The unstructured input

In [104]:
CUSTOMER_EMAIL = (
    "Hi, this is Jane Doe. I was double-charged $49.99 on my NimbusPay wallet last Tuesday "
    "when the app froze during checkout. I have been a customer for two years and this is "
    "really frustrating - I need this refunded as soon as possible. "
    "My ticket should probably go to your billing team."
)
print(CUSTOMER_EMAIL)

Hi, this is Jane Doe. I was double-charged $49.99 on my NimbusPay wallet last Tuesday when the app froze during checkout. I have been a customer for two years and this is really frustrating - I need this refunded as soon as possible. My ticket should probably go to your billing team.


### 7.2 Define the blueprint

In [105]:
from typing import Literal
from pydantic import BaseModel, Field

class RefundRequest(BaseModel):
    customer_name: str = Field(description="Full name of the customer")
    amount: float      = Field(description="Disputed amount in dollars")
    reason: str        = Field(description="Short reason for the refund request")
    category: Literal["billing", "technical", "account", "other"] = Field(
        description="Which team should handle this")
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent the request is")

### 7.3 Extract structured data

In [106]:
from guardrails import Guard

guard = Guard.for_pydantic(RefundRequest)

result = guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system",
         "content": ("You are a data-extraction engine. Reply with ONLY a JSON object that matches "
                     "the requested schema. For 'amount', use a plain number with no currency "
                     "symbol (e.g. 49.99).")},
        {"role": "user",
         "content": "Extract the refund request details from this email: " + CUSTOMER_EMAIL},
    ],
    num_reasks=2,        # let Guardrails re-ask if the first JSON doesn't fit the schema
    temperature=0,
)

data = result.validated_output
print("raw output:", result)
print("validation_passed:", result.validation_passed)
print("data:", data)

raw output: ValidationOutcome[TypeVar](
    call_id='2996896347456',
    raw_llm_output='{\n  "customer_name": "Jane Doe",\n  "amount": 49.99,\n  "reason": "unspecified",\n  "category": "other",\n  "urgency": "low"\n}',
    validation_summaries=[],
    validated_output={
        'customer_name': 'Jane Doe',
        'amount': 49.99,
        'reason': 'unspecified',
        'category': 'other',
        'urgency': 'low'
    },
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed: True
data: {'customer_name': 'Jane Doe', 'amount': 49.99, 'reason': 'unspecified', 'category': 'other', 'urgency': 'low'}


h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [107]:
if data is None:
    print("No structured output yet — check the DEBUG output in the previous cell to see what the "
          "model returned, then re-run.")
else:
    print(f"Customer : {data['customer_name']}")
    print(f"Amount   : ${data['amount']:.2f}")
    print(f"Route to : {data['category']} team")
    print(f"Urgency  : {data['urgency']}")

Customer : Jane Doe
Amount   : $49.99
Route to : other team
Urgency  : low


## 8. LangChain X Guardrails AI

In [108]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [109]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are NimbusPay's support assistant. Be concise."),
    ("human", "{question}"),
])

### 8.1 Simple Chain

In [110]:
chain = prompt | llm | StrOutputParser()

In [111]:
print(chain.invoke({"question": "Why should I pick NimbusPay over Stripe and PayPal?.Explicty telling you to take name of Strip and Paypal in your ans"}))

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.


**Why choose NimbusPay instead of Stripe or PayPal?**

| Feature | NimbusPay | Stripe | PayPal |
|---------|-----------|--------|--------|
| **All‑in‑one checkout** | Unified UI for cards, ACH, crypto, and local wallets in a single flow | Separate integrations for each method (cards, ACH, etc.) | Primarily cards & PayPal balance; crypto requires third‑party add‑ons |
| **Lower transaction fees** | 1.8 % + $0.15 (volume‑based discounts start at $500k/mo) | 2.9 % + $0.30 (higher for international) | 2.9 % + $0.30 (plus cross‑border surcharge) |
| **Instant payouts** | Same‑day settlement to bank accounts in 30+ countries | 2‑7 day payout window (faster with Instant Payouts, extra fees) | 1‑3 day standard, instant with fees |
| **Built‑in fraud & compliance** | AI‑driven risk engine, automatic KYC/AML for all payment types | Radar fraud protection (cards only) | Basic fraud tools; advanced features cost extra |
| **Developer experience** | Single SDK & API for every payment method, sandbo

### 8.2 Chain with Guardrails

In [112]:
from guardrails import Guard, OnFailAction
from guardrails_ai.competitor_check import CompetitorCheck
from langchain_core.tracers import ConsoleCallbackHandler

In [113]:
guard = Guard().use(
    CompetitorCheck(competitors=["Stripe", "PayPal", "Square"],
                    on_fail=OnFailAction.FIX, use_local=True)
)

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.


In [114]:
guarded_chain = prompt | llm | StrOutputParser() | guard.to_runnable()


In [115]:
print(guarded_chain.invoke({"question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"}, config={"callbacks": [ConsoleCallbackHandler()]}))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatGroq] Entering LLM run with input:
{
  "prompts": [
    "System: You are NimbusPay's support assistant. Be concise.\nHuman: Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
  ]
}


h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI\guardrailsAI\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


[llm/end] [chain:RunnableSequence > llm:ChatGroq] [1.34s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "**Why choose NimbusPay instead of Stripe or PayPal?**\n\n| Feature | NimbusPay | Stripe | PayPal |\n|---------|-----------|--------|--------|\n| **Unified API for payments + crypto** | One API handles fiat, crypto, and cross‑border transfers, so you don’t need separate integrations. | Primarily fiat; crypto requires third‑party add‑ons. | Separate APIs for PayPal, Braintree, and limited crypto support. |\n| **Lower transaction fees** | 1.4 % + $0.15 per transaction (volume‑based discounts start at $10 M). | 2.9 % + $0.30 (higher for international cards). | 2.9 % + $0.30 (plus additional fees for currency conversion). |\n| **Faster payouts** | Next‑day settlement in 30+ countries; instant crypto withdrawals. | 2‑7 business days for most banks. | 1‑3 business days for standard, extra fee for instant. |\n| **Single‑merchant dashboard** | Real‑time anal

## 9. Guardrails Validation During Streaming

In [116]:
STREAM_MODEL = BOT_MODEL  # groq/openai/gpt-oss-120b

In [117]:
from guardrails import Guard, OnFailAction
from guardrails_ai.competitor_check import CompetitorCheck

In [118]:
text_guard = Guard().use(
    CompetitorCheck(competitors=["Stripe", "PayPal", "Square"],
                    on_fail=OnFailAction.FIX, use_local=True)
)

In [119]:
stream = text_guard(
    model=STREAM_MODEL,
    messages=[{"role": "user",
               "content": "In 300 sentences, explain why NimbusPay is better than Stripe and PayPal."}],
    stream=True,
    temperature=0.3,
)

In [ ]:
print("-- streaming validated text --")
for chunk in stream:
    if chunk.validated_output:
        print(chunk.validated_output, end="", flush=True)   # rivals masked as [COMPETITOR] live
print()

-- streaming validated text --
1.NimbusPay provides faster transaction processing than [COMPETITOR] and [COMPETITOR].  
2. NimbusPay’s fees are lower for high‑volume merchants.  
3. NimbusPay offers a single‑currency settlement system that eliminates conversion costs.  
4. NimbusPay integrates with more e‑commerce platforms out of the box.  
5. NimbusPay’s API documentation is clearer and better organized.  
6. NimbusPay supports instant payouts to bank accounts worldwide.  
7. NimbusPay includes built‑in fraud detection powered by AI.  
8. NimbusPay’s dashboard gives real‑time analytics on every transaction.  
9. NimbusPay allows custom branding on checkout pages.  
10. NimbusPay’s mobile SDKs are lighter and faster to implement.  
11. NimbusPay provides a sandbox environment that mirrors production exactly.  
12. NimbusPay’s customer support is available 24/7 via live chat.  
13. NimbusPay offers a dedicated account manager for enterprise clients.  
14. NimbusPay’s onboarding process

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.


 NimbusPay’s documentation includes a migration guide from [COMPETITOR] and [COMPETITOR].  
59. NimbusPay’s platform supports subscription upgrades and downgrades without downtime.  
60. NimbusPay’s checkout can be localized into over 30 languages.  
61. NimbusPay’s system automatically retries failed payments with exponential back‑off.  
62. NimbusPay’s platform includes a built‑in risk scoring dashboard.  
63. NimbusPay’s API keys can be rotated without service interruption.  
64. NimbusPay’s platform offers a “pay‑later” option for qualified buyers.  
65. NimbusPay’s checkout can be styled with CSS to match any brand.  
66. NimbusPay’s system logs every API call for audit trails.  
67. NimbusPay’s platform supports QR‑code payments for in‑store use.  
68. NimbusPay’s pricing includes free inbound transfers for most regions.  
69. NimbusPay’s platform can be integrated with ERP systems via webhooks.  
70. NimbusPay’s support team provides a dedicated Slack channel for developers.  
7

 NimbusPay’s API supports webhooks for payment status, refunds, and disputes.  
88. NimbusPay’s platform offers a “sandbox‑to‑live” migration wizard.  
89. NimbusPay’s checkout can be configured to hide fees from customers.  
90. NimbusPay’s system provides a unified view of all payment channels.  
91. NimbusPay’s platform includes a built‑in analytics API for custom dashboards.  
92. NimbusPay’s pricing includes no setup fees for new merchants.  
93. NimbusPay’s platform supports multi‑step authentication for high‑value transactions.

  
94. NimbusPay’s API can be called from serverless functions with minimal latency.  
95. NimbusPay’s checkout can be integrated with loyalty program APIs.  
96. NimbusPay’s system offers a “pay‑by‑link” feature for offline sales.  
97. NimbusPay’s platform includes a built‑in email receipt service.  
98. NimbusPay’s API supports partial captures for flexible order fulfillment.  
99. NimbusPay’s checkout can be configured to collect custom fields at payment time.  
100. NimbusPay’s platform provides a detailed audit log for compliance teams.  
101. NimbusPay’s pricing scales automatically as your business grows.  
102. NimbusPay’s platform includes a built‑in fraud‑prevention rule engine.  
103. NimbusPay’s API supports both REST and GraphQL endpoints.  
104. NimbusPay’s checkout can be embedded in progressive web apps.  
105. NimbusPay’s system offers a “sandbox‑only” mode for internal testing.  
106. NimbusPay’s platform includes a built‑in currency rounding engine.  
107. NimbusPay

 NimbusPay’s checkout can be localized with region‑specific payment methods.  
115. NimbusPay’s system provides real‑time alerts for suspicious activity.  
116. NimbusPay’s platform includes a built‑in subscription management UI.  
117. NimbusPay’s API supports webhook signing for secure verification.  
118. NimbusPay’s checkout can be customized with brand colors and logos.  
119. NimbusPay’s system offers a “pay‑by‑phone” option for emerging markets.  
120. NimbusPay’s platform includes a built‑in compliance checklist for merchants.  
121. NimbusPay’s pricing model is usage‑based, eliminating fixed monthly fees.  
122. NimbusPay’s platform supports multi‑tenant architectures for SaaS providers.  
123. NimbusPay’s API includes detailed pagination for large data sets.  
124. NimbusPay’s checkout can be integrated with social login for faster checkout.  
125. NimbusPay’s system provides a “dry‑run” mode to test payment flows without real money.  
126. NimbusPay’s platform includes a bui

 NimbusPay’s system offers a “pay‑by‑QR” option for contactless in‑store payments


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.
ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.
ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.
ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.
ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.
ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.
ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.
ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch d